# Exploratory data analysis

The goal of this notebook is to profile the data and see if any data cleaning or pre processing steps are required.

## Setting up data from azure

Core idea of this layer: adress all the potential issues flagged by the data profiling library.

In [1]:
import os
import sys
import time
sys.path.append("..")

import pandas as pd
import polars as pl
from data_profiling import ProfileReport
from ingest import get_engine

/Users/aleksandrmedvedev/Desktop/Repositories/olist_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
engine = get_engine()

print("Waking up Azure SQL...")
for attempt in range(1, 11):
    try:
        with engine.connect() as conn:
            conn.exec_driver_sql("SELECT 1")
        print("Database is ready.")
        break
    except Exception:
        print(f"Attempt {attempt}/10 — retrying in 10s...")
        time.sleep(10)
else:
    raise RuntimeError("Database did not respond after 10 attempts.")

Waking up Azure SQL...
Database is ready.


In [3]:
customers            = pl.read_database("SELECT * FROM olist_customers_dataset", engine)
geolocation          = pl.read_database("SELECT * FROM olist_geolocation_dataset", engine)
order_items          = pl.read_database("SELECT * FROM olist_order_items_dataset", engine)
order_payments       = pl.read_database("SELECT * FROM olist_order_payments_dataset", engine)
order_reviews        = pl.read_database("SELECT * FROM olist_order_reviews_dataset", engine)
orders               = pl.read_database("SELECT * FROM olist_orders_dataset", engine)
products             = pl.read_database("SELECT * FROM olist_products_dataset", engine)
sellers              = pl.read_database("SELECT * FROM olist_sellers_dataset", engine)
category_translation = pl.read_database("SELECT * FROM product_category_name_translation", engine)

In [4]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

In [5]:
def generate_table_info(datasets, info_dir="info"):
    os.makedirs(info_dir, exist_ok=True)

    if os.listdir(info_dir):
        print(f"'{info_dir}' is not empty, skipping generation.")
        return

    tables_summary = []

    for name, df in datasets.items():
        lines = [
            f"Dataset: {name}",
            f"Shape: {df.shape[0]} rows x {df.shape[1]} columns",
            "Columns:",
            *(f"  - {col} ({dtype})" for col, dtype in zip(df.columns, df.dtypes)),
        ]
        content = "\n".join(lines)

        with open(f"{info_dir}/{name}.txt", "w") as f:
            f.write(content + "\n")

        tables_summary.append(content)
        print(f"Generated: {info_dir}/{name}.txt")

    with open(f"{info_dir}/tables.txt", "w") as f:
        f.write("\n\n".join(tables_summary) + "\n")

    print(f"Generated: {info_dir}/tables.txt")


generate_table_info(datasets)

'info' is not empty, skipping generation.


In [6]:
def generate_html_reports(datasets, reports_dir="html_reports"):
    os.makedirs(reports_dir, exist_ok=True)

    if os.listdir(reports_dir):
        print(f"'{reports_dir}' is not empty, skipping generation.")
        return

    for name, df in datasets.items():
        report = ProfileReport(df.to_pandas(), title=f"{name} profiling report", minimal=True, progress_bar=False) # minimal generation does not account for duplicates
        report.to_file(f"{reports_dir}/{name}.html")
        print(f"Generated: {reports_dir}/{name}.html")


generate_html_reports(datasets)


'html_reports' is not empty, skipping generation.


## Bronze to Silver

### Customers

Preprocessing: drop customer_unique_id column.

In [7]:
customers.shape

(99441, 5)

In [8]:
customers.head()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
str,str,i64,str,str
"""06b8999e2fba1a1fbc88172c00ba8b…","""861eff4711a542e4b93843c6dd7feb…",14409,"""franca""","""SP"""
"""18955e83d337fd6b2def6b18a428ac…","""290c77bc529b7ac935b93aa66c333d…",9790,"""sao bernardo do campo""","""SP"""
"""4e7b3e00288586ebd08712fdd0374a…","""060e732b5b29e8181a18229c7b0b2b…",1151,"""sao paulo""","""SP"""
"""b2b6027bc5c5109e529d4dc6358b12…","""259dac757896d24d7702b9acbbff3f…",8775,"""mogi das cruzes""","""SP"""
"""4f2d8ab171c80ec8364f7c12e35b23…","""345ecd01c38d18a9036ed96c73b8d0…",13056,"""campinas""","""SP"""


In [9]:
customers['customer_unique_id'].is_unique().all()

False

In [10]:
customers.describe()

statistic,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
str,str,str,f64,str,str
"""count""","""99441""","""99441""",99441.0,"""99441""","""99441"""
"""null_count""","""0""","""0""",0.0,"""0""","""0"""
"""mean""",null,null,35137.474583,null,null
"""std""",null,null,29797.938996,null,null
"""min""","""00012a2ce6f8dcda20d059ce984917…","""0000366f3b9a7992bf8c76cfdf3221…",1003.0,"""abadia dos dourados""","""AC"""
"""25%""",null,null,11347.0,null,null
"""50%""",null,null,24416.0,null,null
"""75%""",null,null,58900.0,null,null
"""max""","""ffffe8b65bbe3087b653a978c870db…","""ffffd2657e2aad2907e67c3e9daecb…",99990.0,"""zortea""","""TO"""


Since the customers_unique_id is actually not unique, we will drop this column.

In [11]:
customers = customers.drop("customer_unique_id")

In [12]:
customers['customer_id'].is_unique().all()

True

### Geolocation

Pre-processing: grouping by the zip code to reduce duplicates.

Geolocation dataset has no unique column, although it is connected via a zip_code_prefix to the customers dataset. We want to make that column unique.

In [13]:
geolocation.shape

(1000163, 5)

In [14]:
geolocation.head()

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
i64,f64,f64,str,str
1037,-23.545621,-46.639292,"""sao paulo""","""SP"""
1046,-23.546081,-46.64482,"""sao paulo""","""SP"""
1046,-23.546129,-46.642951,"""sao paulo""","""SP"""
1041,-23.544392,-46.639499,"""sao paulo""","""SP"""
1035,-23.541578,-46.641607,"""sao paulo""","""SP"""


In [15]:
geolocation.null_count()

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
u32,u32,u32,u32,u32
0,0,0,0,0


In [16]:
geolocation = geolocation.group_by("geolocation_zip_code_prefix").agg(
    pl.col("geolocation_lat").mean(),
    pl.col("geolocation_lng").mean(),
    pl.col("geolocation_city").mode().first(),
    pl.col("geolocation_state").mode().first(),
)

geolocation

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
i64,f64,f64,str,str
74492,-16.677263,-49.357223,"""goiania""","""GO"""
12237,-23.243574,-45.913598,"""sao jose dos campos""","""SP"""
45651,-14.785945,-39.046682,"""ilheus""","""BA"""
88485,-27.895616,-48.931001,"""sao bonifacio""","""SC"""
65021,-2.530786,-44.291551,"""sao luis""","""MA"""
…,…,…,…,…
47803,-12.14708,-44.986898,"""barreiras""","""BA"""
74670,-16.639744,-49.240675,"""goiania""","""GO"""
5588,-23.576017,-46.741186,"""sao paulo""","""SP"""


The aggregation reduced the dataset size from 1M rows to 19K! There were a lot of duplicates in the data, potentially the artifact of order processing. 

In [17]:
geolocation["geolocation_zip_code_prefix"].is_unique().all()

True

In [18]:
geolocation = geolocation.sort("geolocation_zip_code_prefix", descending=False)

### Order items

Pre-processing: conversion to timestamp data

Order item id represents the position of an item within a basket. A single customer can order multiple items e.g 3, so then for that specific order_id we would have 3 order item id's.

In [19]:
order_items.head()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,str,f64,f64
"""00010242fe8c5a6d1ba2dd792cb162…",1,"""4244733e06e7ecb4970a6e2683c13e…","""48436dade18ac8b2bce089ec2a0412…","""2017-09-19 09:45:35""",58.9,13.29
"""00018f77f2f0320c557190d7a144bd…",1,"""e5f2d52b802189ee658865ca93d83a…","""dd7ddc04e1b6c2c614352b383efe2d…","""2017-05-03 11:05:13""",239.9,19.93
"""000229ec398224ef6ca0657da4fc70…",1,"""c777355d18b72b67abbeef9df44fd0…","""5b51032eddd242adc84c38acab88f2…","""2018-01-18 14:48:30""",199.0,17.87
"""00024acbcdf0a6daa1e931b038114c…",1,"""7634da152a4610f1595efa32f14722…","""9d7a1d34a5052409006425275ba1c2…","""2018-08-15 10:10:18""",12.99,12.79
"""00042b26cf59d7ce69dfabb4e55b4f…",1,"""ac6c3623068f30de03045865e4e100…","""df560393f3a51e74553ab94004ba5c…","""2017-02-13 13:57:51""",199.9,18.14


In [20]:
order_items.null_count()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0


In [21]:
order_items = order_items.with_columns(
    pl.col("shipping_limit_date").str.to_datetime("%Y-%m-%d %H:%M:%S")
)

This seems like an intermediate table, that would be useful in the data agreggation within the gold layer.

In [22]:
order_items.head()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""00010242fe8c5a6d1ba2dd792cb162…",1,"""4244733e06e7ecb4970a6e2683c13e…","""48436dade18ac8b2bce089ec2a0412…",2017-09-19 09:45:35,58.9,13.29
"""00018f77f2f0320c557190d7a144bd…",1,"""e5f2d52b802189ee658865ca93d83a…","""dd7ddc04e1b6c2c614352b383efe2d…",2017-05-03 11:05:13,239.9,19.93
"""000229ec398224ef6ca0657da4fc70…",1,"""c777355d18b72b67abbeef9df44fd0…","""5b51032eddd242adc84c38acab88f2…",2018-01-18 14:48:30,199.0,17.87
"""00024acbcdf0a6daa1e931b038114c…",1,"""7634da152a4610f1595efa32f14722…","""9d7a1d34a5052409006425275ba1c2…",2018-08-15 10:10:18,12.99,12.79
"""00042b26cf59d7ce69dfabb4e55b4f…",1,"""ac6c3623068f30de03045865e4e100…","""df560393f3a51e74553ab94004ba5c…",2017-02-13 13:57:51,199.9,18.14


### Order payments

A good sanity check can be to verify that the order sum from order payments matches the price in the order_items & orders tables! 

In [23]:
order_payments.head()

order_id,payment_sequential,payment_type,payment_installments,payment_value
str,i64,str,i64,f64
"""b81ef226f3fe1789b1e8b2acac839d…",1,"""credit_card""",8,99.33
"""a9810da82917af2d9aefd1278f1dcf…",1,"""credit_card""",1,24.39
"""25e8ea4e93396b6fa0d3dd708e76c1…",1,"""credit_card""",1,65.71
"""ba78997921bbcdc1373bb41e913ab9…",1,"""credit_card""",8,107.78
"""42fdf880ba16b47b59251dd489d444…",1,"""credit_card""",2,128.45


In [24]:
order_payments.null_count()

order_id,payment_sequential,payment_type,payment_installments,payment_value
u32,u32,u32,u32,u32
0,0,0,0,0


In [25]:
order_payments.filter(pl.col("payment_sequential") > 5).sort('payment_sequential', descending=True)

order_id,payment_sequential,payment_type,payment_installments,payment_value
str,i64,str,i64,f64
"""fa65dad1b0e818e3ccc5cb0e392313…",29,"""voucher""",1,19.26
"""fa65dad1b0e818e3ccc5cb0e392313…",28,"""voucher""",1,29.05
"""fa65dad1b0e818e3ccc5cb0e392313…",27,"""voucher""",1,66.02
"""ccf804e764ed5650cd8759557269dc…",26,"""voucher""",1,23.1
"""fa65dad1b0e818e3ccc5cb0e392313…",26,"""voucher""",1,28.27
…,…,…,…,…
"""27a940efdd448db29463b53ea0cfa2…",6,"""voucher""",1,4.17
"""4f53648717b48854fb9db283e2cf5d…",6,"""voucher""",1,4.74
"""465c2e1bee4561cb39e0db8c5993aa…",6,"""voucher""",1,13.4


### Order reviews

Pre-processing: drop the review title and the review comment messages, and convert the timestamp data.

In [26]:
order_reviews.head()

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,str,str,str,str
"""7bc2406110b926393aa56f80a40eba…","""73fc7af87114b39712e6da79b0a377…",4,null,null,"""2018-01-18 00:00:00""","""2018-01-18 21:46:59"""
"""80e641a11e56f04c1ad469d5645fdf…","""a548910a1c6147796b98fdf73dbeba…",5,null,null,"""2018-03-10 00:00:00""","""2018-03-11 03:05:13"""
"""228ce5500dc1d8e020d8d1322874b6…","""f9e4b658b201a9f2ecdecbb34bed03…",5,null,null,"""2018-02-17 00:00:00""","""2018-02-18 14:36:24"""
"""e64fb393e7b32834bb789ff8bb3075…","""658677c97b385a9be170737859d351…",5,null,"""Recebi bem antes do prazo esti…","""2017-04-21 00:00:00""","""2017-04-21 22:02:06"""
"""f7c4243c7fe1938f181bec41a392bd…","""8e6bfb81e283fa7e4f11123a3fb894…",5,null,"""Parabéns lojas lannister adore…","""2018-03-01 00:00:00""","""2018-03-02 10:26:53"""


In [27]:
order_reviews['review_id'].is_unique().all()

False

Interestingly not all review id's are completely unique

In [28]:
order_reviews.null_count() / order_reviews.height

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.883415,0.587025,0.0,0.0


More than 50% of the data is missing for the review text columns. Since we are not concerned with NLP we will drop those columns. We will also convert timestamp data into timestamp format.

In [29]:
order_reviews = (
    order_reviews
    .drop(["review_comment_title", "review_comment_message"])
    .with_columns(
        pl.col("review_creation_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
        pl.col("review_answer_timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    )
)
order_reviews

review_id,order_id,review_score,review_creation_date,review_answer_timestamp
str,str,i64,datetime[μs],datetime[μs]
"""7bc2406110b926393aa56f80a40eba…","""73fc7af87114b39712e6da79b0a377…",4,2018-01-18 00:00:00,2018-01-18 21:46:59
"""80e641a11e56f04c1ad469d5645fdf…","""a548910a1c6147796b98fdf73dbeba…",5,2018-03-10 00:00:00,2018-03-11 03:05:13
"""228ce5500dc1d8e020d8d1322874b6…","""f9e4b658b201a9f2ecdecbb34bed03…",5,2018-02-17 00:00:00,2018-02-18 14:36:24
"""e64fb393e7b32834bb789ff8bb3075…","""658677c97b385a9be170737859d351…",5,2017-04-21 00:00:00,2017-04-21 22:02:06
"""f7c4243c7fe1938f181bec41a392bd…","""8e6bfb81e283fa7e4f11123a3fb894…",5,2018-03-01 00:00:00,2018-03-02 10:26:53
…,…,…,…,…
"""574ed12dd733e5fa530cfd4bbf39d7…","""2a8c23fee101d4d5662fa670396eb8…",5,2018-07-07 00:00:00,2018-07-14 17:18:30
"""f3897127253a9592a73be9bdfdf4ed…","""22ec9f0669f784db00fa86d035cf86…",5,2017-12-09 00:00:00,2017-12-11 20:06:42
"""b3de70c89b1510c4cd3d0649fd3024…","""55d4004744368f5571d1f590031933…",5,2018-03-22 00:00:00,2018-03-23 09:10:43


In [30]:
order_reviews.filter(order_reviews['review_id'].is_unique() == False).sort('review_id')

review_id,order_id,review_score,review_creation_date,review_answer_timestamp
str,str,i64,datetime[μs],datetime[μs]
"""00130cbe1f9d422698c812ed8ded19…","""04a28263e085d399c97ae49e0b477e…",1,2018-03-07 00:00:00,2018-03-20 18:08:23
"""00130cbe1f9d422698c812ed8ded19…","""dfcdfc43867d1c1381bfaf62d6b9c1…",1,2018-03-07 00:00:00,2018-03-20 18:08:23
"""0115633a9c298b6a98bcbe4eee7534…","""0c9850b2c179c1ef60d2855e2751d1…",5,2017-09-21 00:00:00,2017-09-26 03:27:47
"""0115633a9c298b6a98bcbe4eee7534…","""78a4201f58af3463bdab842eea4bc8…",5,2017-09-21 00:00:00,2017-09-26 03:27:47
"""0174caf0ee5964646040cd94e15ac9…","""74db91e33b4e1fd865356c89a61abf…",1,2018-03-07 00:00:00,2018-03-08 03:00:53
…,…,…,…,…
"""fe5c833752953fed3209646f1f63b5…","""d3775e436e60258e62e678a0f68a0f…",1,2018-02-28 00:00:00,2018-02-28 13:57:52
"""ff2fc9e68f8aabfbe18d710b83aabd…","""2da58e0a7dcfa4ce1e00fad9d03ca3…",2,2018-03-17 00:00:00,2018-03-19 11:44:15
"""ff2fc9e68f8aabfbe18d710b83aabd…","""1078d496cc6ab9a8e6f2be77abf509…",2,2018-03-17 00:00:00,2018-03-19 11:44:15


Same review can occur for multiple orders! When joining this dataset it is better to use review_id + order_id as a composite key.

### Orders

Pre processing: conversion of the timestamp data to proper type.

In [31]:
orders.head()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,str,str,str,str
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""","""2017-10-02 10:56:33""","""2017-10-02 11:07:15""","""2017-10-04 19:55:00""","""2017-10-10 21:25:13""","""2017-10-18 00:00:00"""
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""","""2018-07-24 20:41:37""","""2018-07-26 03:24:27""","""2018-07-26 14:31:00""","""2018-08-07 15:27:45""","""2018-08-13 00:00:00"""
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""","""2018-08-08 08:38:49""","""2018-08-08 08:55:23""","""2018-08-08 13:50:00""","""2018-08-17 18:06:29""","""2018-09-04 00:00:00"""
"""949d5b44dbf5de918fe9c16f97b45f…","""f88197465ea7920adcdbec7375364d…","""delivered""","""2017-11-18 19:28:06""","""2017-11-18 19:45:59""","""2017-11-22 13:39:59""","""2017-12-02 00:28:42""","""2017-12-15 00:00:00"""
"""ad21c59c0840e6cb83a9ceb5573f81…","""8ab97904e6daea8866dbdbc4fb7aad…","""delivered""","""2018-02-13 21:18:39""","""2018-02-13 22:20:29""","""2018-02-14 19:46:34""","""2018-02-16 18:17:02""","""2018-02-26 00:00:00"""


Order id and customer id are both unique columns

In [32]:
orders["order_id"].is_unique().all() and orders["customer_id"].is_unique().all()

True

There are however some missing values for carrier date and customer date. The delivery pipeline works as follows: 

purchased → payment approved → handed to carrier → delivered to customer → (compared against) estimated delivery date

Missing values are thus informative of underperformance that could be the culprit for the seller under-performance, lowering the sales volume.

If the carrier date is null, then that means that the seller never handed off the delivery to the carrier, while the customer date null means that the order was never delivered to the customer.

In [33]:
orders.filter(pl.col("order_delivered_customer_date").is_null()).select(['order_delivered_carrier_date', 'order_delivered_customer_date'])

order_delivered_carrier_date,order_delivered_customer_date
str,str
null,null
"""2018-06-05 14:32:00""",null
null,null
null,null
"""2018-01-11 19:39:23""",null
…,…
null,null
null,null
null,null


In [34]:
orders.filter(
    pl.col("order_delivered_carrier_date").is_null()
    & pl.col("order_delivered_customer_date").is_not_null()
)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,str,str,str,str
"""2aa91108853cecb43c84a5dc5b2774…","""afeb16c7f46396c0ed54acb45ccaaa…","""delivered""","""2017-09-29 08:52:58""","""2017-09-29 09:07:16""",null,"""2017-11-20 19:44:47""","""2017-11-14 00:00:00"""


Majority of the orders are handled through a carrier, although a single case shows a situation where the order was delivered directly to the customer, bypassing the carrier.

There could be several explanations:

Either the seller delivered the item directly, or this is a logging error. 

In any case a single row will not move meaningfully any aggregate metric, although it is useful to be aware about potential issues.

In [35]:
orders.group_by("order_status").agg(
    pl.col("order_delivered_carrier_date").null_count().alias("carrier_nulls"),
    pl.col("order_delivered_customer_date").null_count().alias("customer_nulls"),
    pl.len().alias("total"),
)

order_status,carrier_nulls,customer_nulls,total
str,u32,u32,u32
"""approved""",2,2,2
"""created""",5,5,5
"""invoiced""",314,314,314
"""delivered""",2,8,96478
"""canceled""",550,619,625
"""unavailable""",609,609,609
"""shipped""",0,1107,1107
"""processing""",301,301,301


We have some minor inconsistencies, which are probably just logging errors, they will not meaningfully affect the results. In general cases the null values are representative - e.g if the order is shipped, it should hot have a customer delivery date. 

In [36]:
orders = orders.with_columns(
    pl.col("order_purchase_timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_approved_at").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_delivered_carrier_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_delivered_customer_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_estimated_delivery_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
)

In [37]:
orders.head()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""",2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""",2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""",2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
"""949d5b44dbf5de918fe9c16f97b45f…","""f88197465ea7920adcdbec7375364d…","""delivered""",2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
"""ad21c59c0840e6cb83a9ceb5573f81…","""8ab97904e6daea8866dbdbc4fb7aad…","""delivered""",2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Products

For the products category in the gold layer it might be better to swap the category names to the english version.

In [38]:
products.head()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,f64,f64,f64,f64,f64,f64,f64
"""1e9e8ef04dbcff4541ed26657ea517…","""perfumaria""",40.0,287.0,1.0,225.0,16.0,10.0,14.0
"""3aa071139cb16b67ca9e5dea641aaa…","""artes""",44.0,276.0,1.0,1000.0,30.0,18.0,20.0
"""96bd76ec8810374ed1b65e29197571…","""esporte_lazer""",46.0,250.0,1.0,154.0,18.0,9.0,15.0
"""cef67bcfe19066a932b7673e239eb2…","""bebes""",27.0,261.0,1.0,371.0,26.0,4.0,26.0
"""9dc1a7de274444849c219cff195d0b…","""utilidades_domesticas""",37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [39]:
products.filter(pl.col('product_category_name').is_null())

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,f64,f64,f64,f64,f64,f64,f64
"""a41e356c76fab66334f36de622ecbd…",null,null,null,null,650.0,17.0,14.0,12.0
"""d8dee61c2034d6d075997acef1870e…",null,null,null,null,300.0,16.0,7.0,20.0
"""56139431d72cd51f19eb9f7dae4d16…",null,null,null,null,200.0,20.0,20.0,20.0
"""46b48281eb6d663ced748f324108c7…",null,null,null,null,18500.0,41.0,30.0,41.0
"""5fb61f482620cb672f5e586bb132ea…",null,null,null,null,300.0,35.0,7.0,12.0
…,…,…,…,…,…,…,…,…
"""b0a0c5dd78e644373b199380612c35…",null,null,null,null,1800.0,30.0,20.0,70.0
"""10dbe0fbaa2c505123c17fdc34a63c…",null,null,null,null,800.0,30.0,10.0,23.0
"""bd2ada37b58ae94cc838b9c0569fec…",null,null,null,null,200.0,21.0,8.0,16.0


In [40]:
products.null_count() / products.height

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.018512,0.018512,0.018512,0.018512,0.000061,0.000061,0.000061,0.000061


Even though we could drop the nulls for the product category name, the unique product_id's might come in handy when joining the tables. We will leave them in the pre processing pipeline.

### Sellers

No preprocesssing required here.

In [41]:
sellers.head()

seller_id,seller_zip_code_prefix,seller_city,seller_state
str,i64,str,str
"""3442f8959a84dea7ee197c632cb2df…",13023,"""campinas""","""SP"""
"""d1b65fc7debc3361ea86b5f14c68d2…",13844,"""mogi guacu""","""SP"""
"""ce3ad9de960102d0677a81f5d0bb7b…",20031,"""rio de janeiro""","""RJ"""
"""c0f3eea2e14555b6faeea3dd58c1b1…",4195,"""sao paulo""","""SP"""
"""51a04a8a6bdcb23deccc82b0b80742…",12914,"""braganca paulista""","""SP"""


### Category translation

No preprocessing required here

In [42]:
category_translation.head()

product_category_name,product_category_name_english
str,str
"""beleza_saude""","""health_beauty"""
"""informatica_acessorios""","""computers_accessories"""
"""automotivo""","""auto"""
"""cama_mesa_banho""","""bed_bath_table"""
"""moveis_decoracao""","""furniture_decor"""


## Silver to Gold - Reshaping the data for business analysis (Power BI)

In [43]:
customers_cleaned            = customers
geolocation_cleaned          = geolocation
order_items_cleaned          = order_items
order_payments_cleaned       = order_payments
order_reviews_cleaned        = order_reviews
orders_cleaned               = orders
products_cleaned             = products
sellers_cleaned              = sellers
category_translation_cleaned = category_translation

Core idea of this layer: remove data redundancy and reshape the data into a star schema to support the business analysis.

In [44]:
# ══════════════════════════════════════════════════════════════════════
# Star schema: one atomic fact (fact_sales) + surrounding dimensions.
# The fact holds ONLY foreign keys + additive measures — every average,
# distribution and rate is left for Power BI to compute from these tables.
# ══════════════════════════════════════════════════════════════════════

In [45]:
# ── fact_sales ── grain: one order item ───────────────────────────────
# order_items is already at item grain; we only bring the order'''s purchase
# date (-> date_key) and customer onto each line so the date and customer
# dimensions can connect. left join keeps every item row, so grain holds.
fact_sales = (
    order_items_cleaned
    .join(
        orders_cleaned.select("order_id", "customer_id", "order_purchase_timestamp"),
        on="order_id",
        how="left",
    )
    .with_columns(
        # integer YYYYMMDD surrogate key into dim_date
        pl.col("order_purchase_timestamp").dt.strftime("%Y%m%d").cast(pl.Int32)
        .alias("order_purchase_date_key"),
        # total amount paid for the line item (item price + shipping fee)
        (pl.col("price") + pl.col("freight_value")).alias("total_cost"),
    )
    .select(
        "order_id",                  # FK -> dim_order  /  FK -> dim_payments
        "seller_id",                 # FK -> dim_seller
        "product_id",                # FK -> dim_product
        "customer_id",               # FK -> dim_customer
        "order_purchase_date_key",   # FK -> dim_date
        "price",                     # measure
        "freight_value",             # measure
        "total_cost",                # measure: price + freight_value
    )
)


In [46]:
fact_sales.head()

order_id,seller_id,product_id,customer_id,order_purchase_date_key,price,freight_value,total_cost
str,str,str,str,i32,f64,f64,f64
"""00010242fe8c5a6d1ba2dd792cb162…","""48436dade18ac8b2bce089ec2a0412…","""4244733e06e7ecb4970a6e2683c13e…","""3ce436f183e68e07877b285a838db1…",20170913,58.9,13.29,72.19
"""00018f77f2f0320c557190d7a144bd…","""dd7ddc04e1b6c2c614352b383efe2d…","""e5f2d52b802189ee658865ca93d83a…","""f6dd3ec061db4e3987629fe6b26e5c…",20170426,239.9,19.93,259.83
"""000229ec398224ef6ca0657da4fc70…","""5b51032eddd242adc84c38acab88f2…","""c777355d18b72b67abbeef9df44fd0…","""6489ae5e4333f3693df5ad4372dab6…",20180114,199.0,17.87,216.87
"""00024acbcdf0a6daa1e931b038114c…","""9d7a1d34a5052409006425275ba1c2…","""7634da152a4610f1595efa32f14722…","""d4eb9395c8c0431ee92fce09860c5a…",20180808,12.99,12.79,25.78
"""00042b26cf59d7ce69dfabb4e55b4f…","""df560393f3a51e74553ab94004ba5c…","""ac6c3623068f30de03045865e4e100…","""58dbd0b2d70206bf40e62cd34e84d7…",20170204,199.9,18.14,218.04


In [47]:
# ── dim_seller ── one row per seller ──────────────────────────────────
dim_seller = sellers_cleaned  # seller_id, zip_code_prefix, city, state

In [48]:
dim_seller.head()

seller_id,seller_zip_code_prefix,seller_city,seller_state
str,i64,str,str
"""3442f8959a84dea7ee197c632cb2df…",13023,"""campinas""","""SP"""
"""d1b65fc7debc3361ea86b5f14c68d2…",13844,"""mogi guacu""","""SP"""
"""ce3ad9de960102d0677a81f5d0bb7b…",20031,"""rio de janeiro""","""RJ"""
"""c0f3eea2e14555b6faeea3dd58c1b1…",4195,"""sao paulo""","""SP"""
"""51a04a8a6bdcb23deccc82b0b80742…",12914,"""braganca paulista""","""SP"""


In [49]:
dim_seller['seller_id'].is_unique().all()

True

In [50]:
# ── dim_product ── one row per product, category translated to English ─
dim_product = (
    products_cleaned
    .join(category_translation_cleaned, on="product_category_name", how="left")
    .select(
        "product_id",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    )
)

In [51]:
dim_product.head()

product_id,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,f64,f64,f64,f64
"""1e9e8ef04dbcff4541ed26657ea517…","""perfumery""",225.0,16.0,10.0,14.0
"""3aa071139cb16b67ca9e5dea641aaa…","""art""",1000.0,30.0,18.0,20.0
"""96bd76ec8810374ed1b65e29197571…","""sports_leisure""",154.0,18.0,9.0,15.0
"""cef67bcfe19066a932b7673e239eb2…","""baby""",371.0,26.0,4.0,26.0
"""9dc1a7de274444849c219cff195d0b…","""housewares""",625.0,20.0,17.0,13.0


In [52]:
dim_product['product_id'].is_unique().all()

True

In [53]:
# ── dim_customer ── geography per customer_id (1:1 with an order) ──────
dim_customer = customers_cleaned.select(
    "customer_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
)

In [54]:
dim_customer.head()

customer_id,customer_zip_code_prefix,customer_city,customer_state
str,i64,str,str
"""06b8999e2fba1a1fbc88172c00ba8b…",14409,"""franca""","""SP"""
"""18955e83d337fd6b2def6b18a428ac…",9790,"""sao bernardo do campo""","""SP"""
"""4e7b3e00288586ebd08712fdd0374a…",1151,"""sao paulo""","""SP"""
"""b2b6027bc5c5109e529d4dc6358b12…",8775,"""mogi das cruzes""","""SP"""
"""4f2d8ab171c80ec8364f7c12e35b23…",13056,"""campinas""","""SP"""


In [55]:
dim_customer['customer_id'].is_unique().all()

True

In [56]:
# ── dim_order ── order-level "why" attributes + one review per order ──
# Reviews are NOT unique per order, so collapse to the latest review score
# first — otherwise the join would fan out (duplicate) the order rows.
order_review = (
    order_reviews_cleaned
    .sort("review_creation_date")
    .group_by("order_id")
    .agg(pl.col("review_score").last())
)
dim_order = (
    orders_cleaned
    .join(order_review, on="order_id", how="left")
    .select(
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "review_score",
    )
)

In [57]:
dim_order.head()

order_id,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score
str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],i64
"""e481f51cbdc54678b7cc49136f2d6a…","""delivered""",2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,4
"""53cdb2fc8bc7dce0b6741e21502734…","""delivered""",2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,4
"""47770eb9100c2d0c44946d9cf07ec6…","""delivered""",2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,5
"""949d5b44dbf5de918fe9c16f97b45f…","""delivered""",2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,5
"""ad21c59c0840e6cb83a9ceb5573f81…","""delivered""",2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,5


In [58]:
dim_order['order_id'].is_unique().all()

True

In [59]:
# ── dim_payments ── one row per order (payment rows rolled up) ────────
# An order can be split across payment methods (payment_sequential 1..n).
# Keep the primary method (sequential = 1) + the total value paid.
dim_payments = (
    order_payments_cleaned
    .sort("payment_sequential")
    .group_by("order_id")
    .agg(
        pl.col("payment_type").first().alias("primary_payment_type"),                 # primary method
        pl.col("payment_installments").first().alias("primary_payment_installments"), # of primary method
        pl.col("payment_value").sum().alias("payment_value"),                 # total across methods
        pl.len().alias("payment_methods_count"),                              # how many methods used
    )
)

In [60]:
dim_payments.head()

order_id,primary_payment_type,primary_payment_installments,payment_value,payment_methods_count
str,str,i64,f64,u32
"""163653bc8aa631e85ac4af4a4f6f35…","""credit_card""",8,84.23,1
"""fe52554f43fb82f60254abfca92f8c…","""credit_card""",1,53.01,1
"""b3224f0819fcb5e1c0f91961c4100a…","""credit_card""",5,82.43,1
"""05e1843fb576fb0a18a1e9bdfa3c18…","""credit_card""",1,64.96,1
"""b8ef769a959add8d51bf27fad025d3…","""boleto""",1,181.11,1


In [61]:
dim_payments['order_id'].is_unique().all()

True

In [62]:
# ── dim_date ── contiguous calendar spanning the order purchase dates ─
# A full date spine (not just the dates that appear) so Power BI can mark
# it as a Date Table and build a clean year > quarter > month hierarchy.
_bounds = orders_cleaned.select(
    pl.col("order_purchase_timestamp").dt.date().min().alias("start"),
    pl.col("order_purchase_timestamp").dt.date().max().alias("end"),
)
dim_date = (
    pl.select(
        pl.date_range(_bounds["start"][0], _bounds["end"][0], interval="1d").alias("full_date")
    )
    .with_columns(
        pl.col("full_date").dt.strftime("%Y%m%d").cast(pl.Int32).alias("date_key"),
        pl.col("full_date").dt.year().alias("year"),
        pl.col("full_date").dt.quarter().alias("quarter"),
        pl.col("full_date").dt.month().alias("month"),
        pl.col("full_date").dt.strftime("%B").alias("month_name"),
    )
    .select("date_key", "full_date", "year", "quarter", "month", "month_name")
)

In [63]:
dim_date.head()

date_key,full_date,year,quarter,month,month_name
i32,date,i32,i8,i8,str
20160904,2016-09-04,2016,3,9,"""September"""
20160905,2016-09-05,2016,3,9,"""September"""
20160906,2016-09-06,2016,3,9,"""September"""
20160907,2016-09-07,2016,3,9,"""September"""
20160908,2016-09-08,2016,3,9,"""September"""


In [64]:
dim_date['date_key'].is_unique().all()

True